# 13b Does the personal map add to MindEye2? Score fusion

**Kaggle:** CPU, Internet on. **Inputs:** the outputs of 10a, 10b and the four 12a notebooks, and 13a.

Development analysis, not a prespecified test. 12b's DINOv2 systems recomputed on the same 1,000 test images and fused with MindEye2's one-session scores
(each query's scores standardised per system, then added; weight 1 primary, a curve from 0.25 to 4). Control:
MindEye2 fused with the person's own ridge model from the same session.

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '13b-mindeye2fusion', 'cpu-dev'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
import time
T0 = time.monotonic()
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
FIXED = json.loads('{"targets": ["subj01", "subj02", "subj05", "subj07"], "K": 200, "folds": 5, "seed": 13, "ridge_seed": 13, "n_subsets": 30, "subset_size": 300, "subset_seed": 401, "basis_images": 3000, "published": {"MindEye2 (1 session)": [79.0, 57.4], "MindTuner (1 session)": [83.1, 76.0], "MindEye2 (full data)": [98.8, 98.3]}, "w_primary": 1.0, "w_curve": [0.25, 0.5, 1.0, 2.0, 4.0]}')
SUBJECTS = ['subj01', 'subj02', 'subj03', 'subj04', 'subj05', 'subj06', 'subj07', 'subj08']

import numpy as np
import pandas as pd
from cross_subject_encoding.eval import lowrank_map as lm
from cross_subject_encoding.eval import nsd_confirm as nc
from cross_subject_encoding.models.ridge import RidgePerOutput
from cross_subject_encoding.utils import config as C

IN = Path("/kaggle/input")
conv = next(p.parent for p in IN.rglob("shared_train_nsd_ids.npy"))
common, cols = {}, {}
for h in ("lh", "rh"):
    masks = [np.load(next((conv / f"{s}_roi").rglob(f"{h}.all-vertices_fsaverage_space.npy"))) for s in SUBJECTS]
    common[h], cl = nc.common_vertices(masks)
    cols[h] = dict(zip(SUBJECTS, cl))
V = sum(len(common[h]) for h in common)
NEW = {}
for s in FIXED["targets"]:
    NEW[s] = dict(np.load(next(IN.rglob(f"12a_{s}.npz"))))
    if not (np.array_equal(NEW[s]["common_lh"], common["lh"]) and np.array_equal(NEW[s]["common_rh"], common["rh"])):
        raise ValueError(f"{s}: 12a used other vertices")
shared_ids = NEW[FIXED["targets"][0]]["shared_ids"]
ids, Y, pool_rows = {}, {}, {}
for s in SUBJECTS:
    ids[s] = np.load(conv / f"{s}_train_nsd_ids.npy")
    pool_rows[s] = np.where(~np.isin(ids[s], shared_ids))[0]  # every shared image out of the donors' training
    Y[s] = np.concatenate([np.load(conv / f"{s}_{h}_fmri.npy", mmap_mode="r")[pool_rows[s]][:, cols[h][s]] for h in ("lh", "rh")], 1)
FEATS = {}
for name in ("alexnet", "dinov2"):
    f = np.load(next(IN.rglob(f"10b_{name}_kpca1000.npz")))
    FEATS[name] = (f["z"].astype(np.float32), {int(i): j for j, i in enumerate(f["nsd_ids"])})
feats = lambda name, nsd_ids: FEATS[name][0][[FEATS[name][1][int(i)] for i in nsd_ids]].astype(np.float64)
for s in FIXED["targets"]:
    missing = [int(i) for i in np.r_[NEW[s]["train_ids"], NEW[s]["test_ids"]] if int(i) not in FEATS["alexnet"][1]]
    if missing:
        raise ValueError(f"{s}: {len(missing)} images without features")
print(V, "vertices | donor pools", {s: len(pool_rows[s]) for s in SUBJECTS}, "| new people",
      {s: (len(NEW[s]["train_ids"]), len(NEW[s]["test_ids"])) for s in FIXED["targets"]}, "|", f"{(time.monotonic() - T0) / 60:.1f} min", flush=True)

In [ ]:
def ols_coef(x, y):
    xa = np.concatenate([np.ones((len(x), 1)), x], 1)
    return np.linalg.solve(xa.T @ xa, xa.T @ np.asarray(y, np.float64)).astype(np.float32)
lin_pred = lambda beta, x: np.concatenate([np.ones((len(x), 1)), x], 1) @ beta.astype(np.float64)
BETA = {}
for f in ("alexnet", "dinov2"):
    for s in SUBJECTS:
        BETA[(f, s)] = ols_coef(feats(f, ids[s][pool_rows[s]]), Y[s])
    print(f, "encoders fitted |", f"{(time.monotonic() - T0) / 60:.1f} min", flush=True)
del Y

In [ ]:
SUBSETS = [np.sort(np.random.default_rng([FIXED["subset_seed"], j]).permutation(n)[:FIXED["subset_size"]])
           for j, n in enumerate([len(NEW[FIXED["targets"][0]]["test_ids"])] * FIXED["n_subsets"])]  # as 12b

def zrows(a):
    a = np.asarray(a, np.float64)
    return (a - a.mean(1, keepdims=True)) / (a.std(1, keepdims=True) + 1e-30)

def oof(fit_predict, k, seed):
    split = np.array_split(np.random.default_rng(seed).permutation(k), FIXED["folds"])
    out = None
    for held in split:
        pred = fit_predict(np.setdiff1d(np.arange(k), held), held)
        if out is None:
            out = np.empty((k, pred.shape[1]))
        out[held] = pred
    return out

def zq(s, axis):
    # standardise each query's scores over the candidates: axis 1 for image retrieval (a brain response's scores over
    # images), axis 0 for brain retrieval (an image's scores over brain responses); rank-preserving per query
    return (s - s.mean(axis, keepdims=True)) / (s.std(axis, keepdims=True) + 1e-12)

def scores(parts):
    # parts: list of (matrix brain x image, weight); fused top-1 in both directions, mean over subsets
    img, brain = [], []
    for sub in SUBSETS:
        mats = [(m[np.ix_(sub, sub)], w) for m, w in parts]
        eye = np.arange(len(sub))
        fi = sum(w * zq(m, 1) for m, w in mats)
        fb = sum(w * zq(m, 0) for m, w in mats)
        img.append((fi.argmax(1) == eye).mean() * 100)
        brain.append((fb.argmax(0) == eye).mean() * 100)
    return float(np.mean(img)), float(np.mean(brain))

pool_all = np.unique(np.concatenate([ids[s][pool_rows[s]] for s in SUBJECTS]))
basis_ids = np.random.default_rng(FIXED["seed"]).choice(pool_all, FIXED["basis_images"], replace=False)  # as 12b
me2_ids = np.load(next(IN.rglob("13a_test_ids.npy")))
rows = []
for s in FIXED["targets"]:
    t = time.monotonic()
    donors = [d for d in SUBJECTS if d != s]
    tr_ids, te_ids = NEW[s]["train_ids"], NEW[s]["test_ids"]
    if not np.array_equal(np.sort(te_ids), me2_ids) or not np.array_equal(te_ids, np.sort(te_ids)):
        raise ValueError(f"{s}: 12a's test images are not 13a's (sorted 73k indices)")
    y_k = NEW[s]["train_y"].astype(np.float64)
    m = zrows(NEW[s]["test_mean"])
    k, d = len(tr_ids), FIXED["seed"]
    xs = {f: (feats(f, tr_ids), feats(f, te_ids)) for f in ("alexnet", "dinov2")}
    def avg_of(x):
        acc = np.zeros((len(x), V))
        for dn in donors:
            acc += lin_pred(BETA[("dinov2", dn)], x)
        return acc / len(donors)
    x_k, x_t = xs["dinov2"]
    a_k, a_t = avg_of(x_k), avg_of(x_t)
    mean, B = lm.prediction_basis(avg_of(feats("dinov2", basis_ids)), FIXED["K"])
    preds = {"borrowing": a_t - a_k.mean(0)}
    model = lm.fit_lowrank_map(a_k, y_k, mean, B, seed=d)
    preds["map"] = lm.apply_lowrank_map(model, a_t, mean, B)
    m_oof = oof(lambda tr, he: lm.apply_lowrank_map(lm.fit_lowrank_map(a_k[tr], y_k[tr], mean, B, seed=d), a_k[he], mean, B), k, d)
    own = {}
    for f in ("alexnet", "dinov2"):
        fx_k, fx_t = xs[f]
        o = RidgePerOutput(lm.ALPHAS, FIXED["folds"], d).fit(fx_k, y_k)
        own[f] = (oof(lambda tr, he: o.refit(fx_k[tr], y_k[tr]).predict(fx_k[he]), k, d), o.predict(fx_t))
    preds["own_ridge"] = own["dinov2"][1]
    w = nc.stack_fit([m_oof, own["alexnet"][0]], y_k)
    preds["mixed"] = nc.stack_apply(w, [m_oof, own["alexnet"][0]], [preds["map"], own["alexnet"][1]], y_k.mean(0))
    del a_k, a_t, m_oof, own
    R = {name: (m @ zrows(p).T / m.shape[1]) for name, p in preds.items()}  # Pearson, brain x image, as 12b
    S = np.load(next(IN.rglob(f"13a_{s}_cosine.npy"))).astype(np.float64)
    def add(system, parts):
        img, brain = scores(parts)
        rows.append({"subject": s, "system": system, "image": img, "brain": brain})
    add("mindeye2", [(S, 1.0)])
    for name, r in R.items():
        add(name, [(r, 1.0)])
        for wt in FIXED["w_curve"]:
            add(f"mindeye2+{name}|w={wt}", [(S, 1.0), (r, wt)])
    pd.DataFrame(rows).to_csv(C.METRICS_DIR / "13b_scores.csv", index=False)
    print(s, {r["system"]: (round(r["image"], 1), round(r["brain"], 1)) for r in rows if r["subject"] == s and ("|" not in r["system"] or r["system"].endswith(f"w={FIXED['w_primary']}"))},
          f"| {time.monotonic() - t:.0f} s |", f"{(time.monotonic() - T0) / 60:.1f} min", flush=True)

In [ ]:
frame = pd.DataFrame(rows)
res = {"post_hoc": "development: score fusion of MindEye2's one-session model with our systems", "systems": {}, "contrasts": {}}
for system, g in frame.groupby("system", sort=False):
    g = g.set_index("subject").reindex(FIXED["targets"])
    res["systems"][system] = {"image": float(g["image"].mean()), "brain": float(g["brain"].mean()),
                              "per_person_image": g["image"].round(2).to_dict(), "per_person_brain": g["brain"].round(2).to_dict()}
def diff(a, b, m):
    x = (frame[frame.system == a].set_index("subject")[m] - frame[frame.system == b].set_index("subject")[m]).reindex(FIXED["targets"])
    return {"mean": float(x.mean()), "positive": int((x > 0).sum()), "n": int(x.notna().sum())}
wp = FIXED["w_primary"]
for m in ("image", "brain"):
    for name in ("mixed", "map", "own_ridge", "borrowing"):
        res["contrasts"][f"mindeye2+{name}_minus_mindeye2|{m}"] = diff(f"mindeye2+{name}|w={wp}", "mindeye2", m)
    res["contrasts"][f"mindeye2+mixed_minus_mindeye2+own_ridge|{m}"] = diff(f"mindeye2+mixed|w={wp}", f"mindeye2+own_ridge|w={wp}", m)
res["check_against_12b"] = {"mixed": [res["systems"]["mixed"]["image"], res["systems"]["mixed"]["brain"]], "12b": [64.5, 34.2]}
(C.METRICS_DIR / "13b_summary.json").write_text(json.dumps(res, indent=2))
print("13B_SUMMARY_JSON " + json.dumps(res))